# RAG dengan LLM Open Source Lokal: Llama 3.1 8B + nomic-embed

Notebook ini membangun RAG yang berjalan sepenuhnya lokal di GPU Colab dan mengujinya dengan set pertanyaan standar yang kategorinya sama dengan notebook RAG API (gpt-4o-mini).

**Stack:**
- LLM: Llama 3.1 8B fine-tune bahasa Indonesia (`rubythalib33/llama3_1_8b_finetuned_bahasa_indonesia`, Q4_K_M) via llama-cpp-python
- Embedding: `nomic-embed-text-v1.5` (GGUF, 768 dimensi)
- Retrieval: cosine similarity dengan NumPy

## 1. Install llama-cpp-python dengan CUDA

`pip install llama-cpp-python` biasa mem-build versi **CPU saja**. Di versi awal notebook ini, akibatnya `n_gpu_layers=-1` diabaikan diam-diam dan semua layer berjalan di CPU (sekitar 1,55 token/detik). Flag `GGML_CUDA=on` membuat build yang bisa memakai GPU.

Build dari source dengan CUDA memakan waktu cukup lama (bisa 10–20 menit di Colab). Pastikan runtime sudah GPU (T4) sebelum menjalankan sel ini.

In [1]:
!CMAKE_ARGS="-DGGML_CUDA=on" pip install -q llama-cpp-python --no-cache-dir

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import llama_cpp
gpu_ok = llama_cpp.llama_supports_gpu_offload()
print("llama-cpp-python", llama_cpp.__version__, "| GPU offload:", gpu_ok)
assert gpu_ok, "Build tanpa CUDA. Ulangi sel install di atas dengan runtime GPU."

name, memory.total [MiB]
NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB


llama-cpp-python 0.3.35 | GPU offload: True


## 2. Load Model & Embedder

- `n_ctx=4096`: tanpa parameter ini, konteks default hanya 512 token. Di versi awal, prompt 259 token ditambah `max_tokens=256` sudah melewati batas itu.
- `verbose=False`: menyembunyikan log llama.cpp yang sangat panjang. GPU sudah dicek di sel sebelumnya.

In [3]:
from llama_cpp import Llama
import numpy as np

llm = Llama.from_pretrained(
    repo_id="rubythalib33/llama3_1_8b_finetuned_bahasa_indonesia",
    filename="unsloth.Q4_K_M.gguf",
    n_gpu_layers=-1,
    n_ctx=4096,
    verbose=False,
)

text_embedder = Llama.from_pretrained(
    repo_id="nomic-ai/nomic-embed-text-v1.5-GGUF",
    filename="nomic-embed-text-v1.5.Q4_K_M.gguf",
    embedding=True,
    n_gpu_layers=-1,
    verbose=False,
)
print("Model dan embedder siap.")

Model dan embedder siap.


Model ini di-fine-tune dengan format **Alpaca**, jadi dipanggil sebagai raw completion dengan template di bawah. Memakai `create_chat_completion` justru membuat llama-cpp jatuh ke template Llama 2 (`Using fallback chat format: llama-2`), karena GGUF ini tidak membawa chat template.

In [4]:
alpaca_prompt = """Di bawah ini adalah instruksi yang menjelaskan tugas, dipasangkan dengan masukan yang memberikan konteks lebih lanjut. Tulis tanggapan yang melengkapi permintaan dengan tepat.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

## 3. Embedding dengan Prefix Task

nomic-embed-text-v1.5 dilatih dengan prefix yang membedakan dokumen dan query:
- `search_document: ` untuk teks yang disimpan
- `search_query: ` untuk pertanyaan user

Tanpa prefix, model tetap menghasilkan vektor, tapi kualitas retrieval-nya turun. Vektor dinormalisasi supaya cosine similarity cukup dihitung dengan dot product.

Embedding dilakukan satu per satu. Untuk model encoder seperti nomic, cara ini paling aman di berbagai versi llama-cpp-python, dan untuk 20 dokumen waktunya tetap singkat.

In [5]:
def embed(texts, kind):
    prefix = {"doc": "search_document: ", "query": "search_query: "}[kind]
    vecs = []
    for t in texts:
        out = text_embedder.create_embedding(prefix + t)
        vecs.append(out["data"][0]["embedding"])
    vecs = np.array(vecs, dtype=np.float32)
    return vecs / np.linalg.norm(vecs, axis=1, keepdims=True)

## 4. Knowledge Base & Index

In [6]:
DOCUMENTS = [
    "Lari adalah aktivitas olahraga kardiovaskular yang melibatkan gerakan berulang kaki untuk berpindah dengan cepat dari satu titik ke titik lain.",
    "Olahraga lari bermanfaat untuk meningkatkan kesehatan jantung, memperbaiki sirkulasi darah, dan menurunkan risiko penyakit kronis seperti diabetes dan hipertensi.",
    "Lari secara rutin dapat membantu menjaga berat badan ideal karena membakar kalori secara efektif dan meningkatkan metabolisme tubuh.",
    "Terdapat beberapa jenis lari seperti jogging, sprint, lari jarak pendek, lari jarak menengah, lari jarak jauh, dan trail running yang bisa dipilih sesuai tujuan latihan.",
    "Sebelum mulai lari, pemanasan sangat penting dilakukan untuk mengurangi risiko cedera otot dan mempersiapkan tubuh terhadap aktivitas intens.",
    "Pendinginan setelah lari membantu menurunkan detak jantung secara bertahap dan mengurangi kekakuan otot pasca latihan.",
    "Pemilihan sepatu lari yang tepat, dengan cushioning dan support yang sesuai, sangat berpengaruh terhadap kenyamanan dan pencegahan cedera.",
    "Teknik lari yang baik meliputi posisi tubuh tegak, langkah yang efisien, dan ritme napas yang teratur untuk menjaga stamina selama berlari.",
    "Latihan lari dapat disusun dalam bentuk program bertahap, misalnya kombinasi jalan–lari untuk pemula hingga program khusus untuk persiapan lomba 5K, 10K, half marathon, atau marathon.",
    "Interval training dalam lari melibatkan pergantian antara lari cepat dan lari lambat atau jalan, yang efektif untuk meningkatkan kecepatan dan kapasitas VO2 max.",
    "Latihan lari jarak jauh (long run) dilakukan dengan intensitas sedang dan durasi lebih panjang untuk membangun daya tahan (endurance).",
    "Nutrisi sebelum lari sebaiknya mengandung karbohidrat ringan dan mudah dicerna, sedangkan setelah lari dianjurkan kombinasi protein dan karbohidrat untuk pemulihan otot.",
    "Hidrasi yang cukup sebelum, selama, dan setelah lari penting untuk mencegah dehidrasi dan menjaga performa tubuh.",
    "Cedera umum pada pelari meliputi shin splints, cedera lutut, nyeri tumit (plantar fasciitis), dan tegang otot paha belakang, yang sering disebabkan oleh overtraining atau teknik yang kurang tepat.",
    "Istirahat dan hari tanpa latihan (rest day) merupakan bagian penting dari program lari untuk memberi waktu pemulihan bagi tubuh dan mencegah overtraining.",
    "Menggunakan aplikasi atau jam tangan lari dapat membantu memantau jarak, pace, detak jantung, dan progres latihan secara berkala.",
    "Latihan kekuatan (strength training) seperti squat, lunges, dan core training dapat mendukung performa lari dengan memperkuat otot pendukung dan meningkatkan stabilitas tubuh.",
    "Menentukan tujuan lari yang jelas, seperti menyelesaikan lomba pertama atau meningkatkan pace tertentu, dapat menjadi motivasi tambahan untuk konsistensi latihan.",
    "Lari berkelompok atau bersama komunitas lari dapat meningkatkan motivasi, memberikan dukungan sosial, dan membuat latihan terasa lebih menyenangkan.",
    "Kualitas tidur yang baik dan pola hidup sehat secara keseluruhan sangat berpengaruh terhadap performa lari dan pemulihan setelah latihan intensif.",
]

DOC_EMB = embed(DOCUMENTS, "doc")
print(f"Dokumen terindeks: {DOC_EMB.shape[0]} | dimensi: {DOC_EMB.shape[1]}")

Dokumen terindeks: 20 | dimensi: 768


## 5. Retrieval & Visualisasi Skor

In [7]:
def search(query, top_k=3):
    scores = DOC_EMB @ embed([query], "query")[0]
    order = np.argsort(-scores)[:top_k]
    return [{"idx": int(i), "text": DOCUMENTS[i], "score": float(scores[i])} for i in order]

def show_retrieval_scores(question, top_k=3):
    results = search(question, top_k=len(DOCUMENTS))
    print(f'Retrieval: "{question}"')
    for rank, r in enumerate(results, 1):
        bar = "█" * int(max(r["score"], 0) * 40)
        tag = "DIPAKAI" if rank <= top_k else "skip   "
        print(f"{tag} #{rank:<2} {r['score']:.4f}  {r['text'][:45]:<45} {bar}")
    print()

show_retrieval_scores("apa itu Rag dan bagaimana cara kerjanya")
show_retrieval_scores("Apa manfaat interval training?")

Retrieval: "apa itu Rag dan bagaimana cara kerjanya"
DIPAKAI #1  0.6859  Olahraga lari bermanfaat untuk meningkatkan k ███████████████████████████
DIPAKAI #2  0.6848  Istirahat dan hari tanpa latihan (rest day) m ███████████████████████████
DIPAKAI #3  0.6765  Lari adalah aktivitas olahraga kardiovaskular ███████████████████████████
skip    #4  0.6746  Menggunakan aplikasi atau jam tangan lari dap ██████████████████████████
skip    #5  0.6665  Sebelum mulai lari, pemanasan sangat penting  ██████████████████████████
skip    #6  0.6664  Pemilihan sepatu lari yang tepat, dengan cush ██████████████████████████
skip    #7  0.6628  Nutrisi sebelum lari sebaiknya mengandung kar ██████████████████████████
skip    #8  0.6626  Teknik lari yang baik meliputi posisi tubuh t ██████████████████████████
skip    #9  0.6609  Latihan lari dapat disusun dalam bentuk progr ██████████████████████████
skip    #10 0.6541  Cedera umum pada pelari meliputi shin splints ██████████████████████████
skip    #11 0.

## 6. RAG: Retrieve + Generate

Perubahan dari versi awal:
- **Threshold opsional**, sama seperti di notebook API.
- **Kalimat penolakan eksplisit.** Model kecil lebih patuh kalau diberi kalimat yang persis, bukan sekadar "jawab jujur bahwa kamu tidak tahu".
- **Stop sequence** ditambah token akhir Llama 3.1 (`<|eot_id|>`, `<|end_of_text|>`).
- Fungsi mengembalikan skor, token, latensi, dan token/detik.

In [8]:
import time

FALLBACK = "Maaf, informasi tersebut tidak ada di knowledge base."

INSTRUCTION = (
    "Gunakan HANYA konteks yang diberikan untuk menjawab pertanyaan pengguna secara singkat. "
    "Jangan menambahkan informasi dari luar konteks. "
    f"Jika jawabannya tidak ada di konteks, jawab persis: \"{FALLBACK}\""
)

def build_rag_prompt(user_query, retrieved_texts):
    context_block = "\n\n---\n\n".join(retrieved_texts)
    input_section = (f"Konteks berikut diambil dari knowledge base:\n\n{context_block}\n\n"
                     f"Pertanyaan pengguna:\n{user_query}\n")
    return alpaca_prompt.format(INSTRUCTION, input_section, "")

def rag_answer(question, top_k=3, threshold=None, max_tokens=256, temperature=0.2, verbose=True):
    t0 = time.perf_counter()
    retrieved = search(question, top_k=top_k)
    used = [r for r in retrieved if threshold is None or r["score"] >= threshold]

    usage, gen_time = None, None
    if not used:
        answer = FALLBACK
    else:
        prompt = build_rag_prompt(question, [r["text"] for r in used])
        t_gen = time.perf_counter()
        out = llm(prompt, max_tokens=max_tokens, temperature=temperature,
                  stop=["### Instruction:", "### Input:", "### Response:", "<|eot_id|>", "<|end_of_text|>"])
        gen_time = time.perf_counter() - t_gen
        answer = out["choices"][0]["text"].strip()
        usage = out["usage"]
    latency = time.perf_counter() - t0
    tok_s = round(usage["completion_tokens"] / gen_time, 2) if usage and gen_time else None

    if verbose:
        print("=" * 70)
        print("PERTANYAAN:", question)
        for r in retrieved:
            tanda = "dipakai" if r in used else "dibuang"
            print(f"  [{tanda}] {r['score']:.4f} | {r['text'][:70]}")
        print("\nJAWABAN:\n" + answer)
        info = f"latensi {latency:.2f}s"
        if usage:
            info += (f" | token prompt {usage['prompt_tokens']}, jawaban {usage['completion_tokens']}"
                     f" | {tok_s} token/detik (termasuk evaluasi prompt)")
        else:
            info += " | LLM tidak dipanggil"
        print(f"({info})\n")

    return {
        "pertanyaan": question,
        "top_score": round(retrieved[0]["score"], 4),
        "dok_dipakai": len(used),
        "llm_dipanggil": usage is not None,
        "jawaban": answer,
        "latensi_s": round(latency, 2),
        "prompt_tokens": usage["prompt_tokens"] if usage else 0,
        "completion_tokens": usage["completion_tokens"] if usage else 0,
        "token_per_detik": tok_s,
    }

Panggilan pertama ke model biasanya lebih lambat (inisialisasi kernel GPU). Satu panggilan pemanasan dijalankan dulu supaya angka latensi di evaluasi tidak terdistorsi.

In [9]:
_ = llm("Halo", max_tokens=8)
print("Pemanasan selesai.")

Pemanasan selesai.


## 7. Evaluasi dengan Set Pertanyaan Standar

Kategori yang sama dengan notebook API, dengan pertanyaan yang disesuaikan ke knowledge base olahraga lari. Pertanyaan "apa itu Rag" dan "halo siapa kamu" adalah pertanyaan yang sama dengan run versi awal.

In [10]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

QUESTIONS = [
    ("dalam domain", "Apa manfaat interval training?"),
    ("majemuk", "apa itu lari dan apa saja jenis-jenisnya?"),
    ("luar domain (istilah AI)", "apa itu Rag dan bagaimana cara kerjanya"),
    ("luar domain (jauh)", "Berapa harga saham Tesla hari ini?"),
    ("sapaan", "halo siapa kamu"),
]

def run_eval(threshold, label):
    rows = []
    for kategori, q in QUESTIONS:
        r = rag_answer(q, threshold=threshold, verbose=True)
        rows.append({"mode": label, "kategori": kategori, **r})
    return pd.DataFrame(rows)

df_tanpa = run_eval(threshold=None, label="tanpa threshold")
df_tanpa[["kategori", "top_score", "llm_dipanggil", "latensi_s", "completion_tokens", "token_per_detik"]]

PERTANYAAN: Apa manfaat interval training?
  [dipakai] 0.7226 | Interval training dalam lari melibatkan pergantian antara lari cepat d
  [dipakai] 0.6179 | Istirahat dan hari tanpa latihan (rest day) merupakan bagian penting d
  [dipakai] 0.6146 | Latihan kekuatan (strength training) seperti squat, lunges, dan core t

JAWABAN:
Interval training dalam lari dapat meningkatkan kecepatan dan kapasitas VO2 max.
(latensi 0.18s | token prompt 284, jawaban 20 | 111.93 token/detik (termasuk evaluasi prompt))

PERTANYAAN: apa itu lari dan apa saja jenis-jenisnya?
  [dipakai] 0.7134 | Olahraga lari bermanfaat untuk meningkatkan kesehatan jantung, memperb
  [dipakai] 0.7056 | Latihan lari dapat disusun dalam bentuk program bertahap, misalnya kom
  [dipakai] 0.7044 | Menggunakan aplikasi atau jam tangan lari dapat membantu memantau jara

JAWABAN:
Lari adalah bentuk olahraga yang melibatkan berjalan atau berlari dengan kecepatan tinggi.  Jenis lari yang umum termasuk lari jarak pendek, lari jarak me

,kategori,top_score,llm_dipanggil,latensi_s,completion_tokens,token_per_detik
0,dalam domain,0.7226,True,0.18,20,111.93
1,majemuk,0.7134,True,0.49,89,181.80
2,luar domain (istilah AI),0.6859,True,0.23,40,175.83
3,luar domain (jauh),0.6979,True,0.09,12,142.22
4,sapaan,0.5989,True,0.09,12,138.76


### Kalibrasi threshold

Skala skor nomic berbeda dengan `text-embedding-3-small`, jadi threshold **tidak bisa disalin** dari notebook API. Lihat kolom `top_score` di atas, lalu isi `THRESHOLD` dengan nilai di antara skor tertinggi pertanyaan luar domain dan skor terendah pertanyaan dalam domain.

In [12]:
THRESHOLD = 0.7  # isi setelah melihat tabel di atas, misalnya 0.5
assert THRESHOLD is not None, "Isi THRESHOLD dulu berdasarkan kolom top_score."

df_dengan = run_eval(threshold=THRESHOLD, label=f"threshold {THRESHOLD}")
df_dengan[["kategori", "top_score", "dok_dipakai", "llm_dipanggil", "latensi_s", "completion_tokens"]]

PERTANYAAN: Apa manfaat interval training?
  [dipakai] 0.7226 | Interval training dalam lari melibatkan pergantian antara lari cepat d
  [dibuang] 0.6179 | Istirahat dan hari tanpa latihan (rest day) merupakan bagian penting d
  [dibuang] 0.6146 | Latihan kekuatan (strength training) seperti squat, lunges, dan core t

JAWABAN:
Interval training dalam lari dapat meningkatkan kecepatan dan kapasitas VO2 max.
(latensi 0.12s | token prompt 195, jawaban 20 | 177.93 token/detik (termasuk evaluasi prompt))

PERTANYAAN: apa itu lari dan apa saja jenis-jenisnya?
  [dipakai] 0.7134 | Olahraga lari bermanfaat untuk meningkatkan kesehatan jantung, memperb
  [dipakai] 0.7056 | Latihan lari dapat disusun dalam bentuk program bertahap, misalnya kom
  [dipakai] 0.7044 | Menggunakan aplikasi atau jam tangan lari dapat membantu memantau jara

JAWABAN:
Lari adalah olahraga yang melibatkan berjalan kaki dengan kecepatan tinggi.  Jenis lari yang umum termasuk lari jarak pendek, lari jarak menengah, lari ja

,kategori,top_score,dok_dipakai,llm_dipanggil,latensi_s,completion_tokens
0,dalam domain,0.7226,1,True,0.12,20
1,majemuk,0.7134,3,True,0.36,65
2,luar domain (istilah AI),0.6859,0,False,0.00,0
3,luar domain (jauh),0.6979,0,False,0.00,0
4,sapaan,0.5989,0,False,0.00,0


## 8. Simpan Hasil Evaluasi

In [13]:
hasil = pd.concat([df_tanpa, df_dengan], ignore_index=True)
hasil.insert(0, "sistem", "Lokal (Llama 3.1 8B Q4_K_M, GPU)")
hasil.to_csv("hasil_rag_lokal.csv", index=False)
print(f"Tersimpan: hasil_rag_lokal.csv ({len(hasil)} baris)")
hasil[["mode", "kategori", "top_score", "llm_dipanggil", "latensi_s", "token_per_detik"]]

Tersimpan: hasil_rag_lokal.csv (10 baris)


,mode,kategori,top_score,llm_dipanggil,latensi_s,token_per_detik
0,tanpa threshold,dalam domain,0.7226,True,0.18,111.93
1,tanpa threshold,majemuk,0.7134,True,0.49,181.80
2,tanpa threshold,luar domain (istilah AI),0.6859,True,0.23,175.83
3,tanpa threshold,luar domain (jauh),0.6979,True,0.09,142.22
4,tanpa threshold,sapaan,0.5989,True,0.09,138.76
5,threshold 0.7,dalam domain,0.7226,True,0.12,177.93
6,threshold 0.7,majemuk,0.7134,True,0.36,179.26
7,threshold 0.7,luar domain (istilah AI),0.6859,False,0.00,NaN
8,threshold 0.7,luar domain (jauh),0.6979,False,0.00,NaN
9,threshold 0.7,sapaan,0.5989,False,0.00,NaN


## Ringkasan

- llama-cpp-python harus di-build dengan CUDA supaya `n_gpu_layers` benar-benar memindahkan layer ke GPU. Build CPU tidak memberi error, hanya lambat.
- `n_ctx` perlu diset eksplisit, karena default 512 token terlalu kecil untuk prompt RAG.
- Model ini dipanggil sebagai raw completion format Alpaca, sesuai format fine-tune-nya.
- Prefix `search_query: ` / `search_document: ` adalah bagian dari cara pakai nomic-embed, bukan opsional.
- Model 8B terkuantisasi lebih mudah "dipaksa" menjawab oleh konteks yang tidak relevan, sehingga threshold di tahap retrieval jadi pertahanan utama, bukan hanya instruksi di prompt.

---
*Hands-on rubythalib.ai.*